# Metadata, PII Scrubbing, and Pipeline Wiring

### 1) Objective
To demonstrate what makes a Retrieval-Augmented Generation (RAG) pipeline production-ready: applying **metadata** (for filtered retrieval) and **PII scrubbing** (so sensitive data like KinkyShoeBros employee details never reaches the vector store). We wrap by wiring everything into a mini-pipeline on one sample document.

### 2) Libraries Used
- **Qdrant (`qdrant-client`)**: A vector database used to store embeddings and perform metadata-filtered searches.
- **OpenAI / Ollama**: For generating vector embeddings.
- **Presidio (`presidio-analyzer`, `presidio-anonymizer`)**: Microsoft's NLP tool backed by `spaCy` for high-recall Named Entity Recognition (NER) to detect and redact PII.
- **PyMuPDF / python-docx**: For document parsing.

### 3) Ambition of the Code
To ensure that ingested enterprise data (like internal shoe allocation policies) is both highly searchable (via metadata filters) and strictly secure (via multi-layered PII anonymization) before it is ever committed to a search index.

---

### Understanding the Code & Concepts

**1) What the code expects to do:**
It installs dependencies, defines a strict 9-field metadata schema, runs a filtered search demo in Qdrant, shows why leaking PII into a vector DB is dangerous, layers Regex and Presidio AI to scrub sensitive data, and wires it all together into an `ingest_one_pdf` function.

**2) What the outputs mean:**
You will see chunks successfully filtered by date or document type. You will also see raw text containing fake HR data (like emails and employee IDs) systematically replaced with tags like `<EMAIL_ADDRESS>` or `<PERSON>` before being ingested.

**3) What it means to scrub PII:**
PII (Personally Identifiable Information) scrubbing is the process of detecting sensitive entities (names, phones, IDs) and replacing them with anonymized placeholders. Because vector databases are search indexes, any unscrubbed PII becomes permanently discoverable by users querying the system.

**4) What it means to use Metadata Filters:**
Metadata filters allow the AI to restrict vector searches to specific subsets of data (e.g., "Only search documents from 2026"). This prevents outdated or irrelevant chunks from corrupting the AI's answer, even if they are semantically similar to the user's question.

---

## Cell 1 — Setup & Dependency Installations

First, we must install our vector database client, embedding tools, and Presidio (which requires downloading a spaCy language model).

In [1]:
%pip install qdrant-client openai pymupdf
%pip install presidio-analyzer presidio-anonymizer spacy
!python -m spacy download en_core_web_sm

Note: you may need to restart the kernel to use updated packages.
  Using cached pyyaml-6.0.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 47.8 MB/s eta 0:00:00
Using cached pyyaml-6.0.3-cp313-cp313-macosx_11_0_arm64.whl (173 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 683.8/683.8 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.1/788.1 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 58.0 MB/s eta 0:00:00
  Attempting uninstall: pyyaml
    Found existing installation: PyYAML 6.0.2
    Uninstalling PyYAML-6.0.2:
      Successfully uninstalled PyYAML-6.0.2
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2

In [2]:
import os
import re
from pathlib import Path
import pymupdf

# ── Define KinkyShoeBros sample paths ──
SAMPLE_DIR = Path.cwd() / "sample_docs"
SAMPLE_PDF  = SAMPLE_DIR / "kinkyshoebros_policies.pdf"
SAMPLE_HTML = SAMPLE_DIR / "kinkyshoebros_catalog.html"
SAMPLE_DOCX = SAMPLE_DIR / "kinkyshoebros_preloved.docx"

for path in [SAMPLE_PDF, SAMPLE_HTML, SAMPLE_DOCX]:
    assert path.exists(), f"Missing {path.name}. Please generate the documents first."

# ── Define Core Helpers (Normally imported from a pipeline script) ──
METADATA_FIELDS = ["chunk_id", "source", "doc_type", "section_path", "page", "date", "language", "version", "ingested_at"]

def parse_pdf(path: Path) -> str:
    doc = pymupdf.open(path)
    text = "\n\n".join(page.get_text() for page in doc)
    doc.close()
    return text

def chunk_recursive(text: str, max_size=400) -> list[str]:
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
    chunks = []
    for para in paragraphs:
        if len(para) <= max_size:
            chunks.append(para)
        else:
            sentences = re.split(r'(?<=[.!?])\s+', para)
            current = ""
            for sent in sentences:
                if len(current) + len(sent) + 1 <= max_size:
                    current = (current + " " + sent).strip()
                else:
                    if current: chunks.append(current)
                    current = sent
            if current: chunks.append(current)
    return chunks

print("Setup complete. Sample documents and helpers loaded.")

Setup complete. Sample documents and helpers loaded.


---

## Cell 2 — The 9-field metadata schema

Every chunk gets 9 pieces of metadata attached. The schema is standard — once you commit to it, downstream code (retrieval, filtering, observability) can rely on it.

Let's build a chunk with all 9 fields on the KinkyShoeBros PDF.

In [3]:
from datetime import datetime, timezone

print("Metadata schema — 9 fields per chunk:")
for i, field in enumerate(METADATA_FIELDS, 1):
    print(f"  {i}. {field}")

pdf_text = parse_pdf(SAMPLE_PDF)
chunks = chunk_recursive(pdf_text, max_size=400)
first_chunk_text = chunks[0]

chunk_with_metadata = {
    "chunk_id":     "kinkyshoebros_policies#0",
    "source":       SAMPLE_PDF.name,
    "doc_type":     "pdf",
    "section_path": "Employee Handbook > Shoe Allocation", 
    "page":         1,
    "date":         "2026-01-15",
    "language":     "en",
    "version":      "v1.0",
    "ingested_at":  datetime.now(timezone.utc).isoformat(),
    "text":         first_chunk_text,
}

print("\n══ A real chunk with all 9 metadata fields ══\n")
for k, v in chunk_with_metadata.items():
    if k == "text":
        v = v[:80] + "..."
    print(f"  {k:15s}  {v!r}")

Metadata schema — 9 fields per chunk:
  1. chunk_id
  2. source
  3. doc_type
  4. section_path
  5. page
  6. date
  7. language
  8. version
  9. ingested_at

══ A real chunk with all 9 metadata fields ══

  chunk_id         'kinkyshoebros_policies#0'
  source           'kinkyshoebros_policies.pdf'
  doc_type         'pdf'
  section_path     'Employee Handbook > Shoe Allocation'
  page             1
  date             '2026-01-15'
  language         'en'
  version          'v1.0'
  ingested_at      '2026-10-10T23:02:53.414569+00:00'
  text             'kinkyShoeBros — Internal Employee Handbook\nSection 1: Employee Footwear Allocati...'


---

## Cell 3 — Qdrant filter DSL

Metadata is only useful if you can filter on it at query time. Let's build a local Qdrant in-memory collection with 6 chunks (mixed doc types + dates), then run **filtered queries**.

In [4]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue, Range
import numpy as np

# Using local in-memory Qdrant for the demo
qdrant = QdrantClient(":memory:")
COLLECTION = "kinkyshoebros_demo"

qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=3, distance=Distance.COSINE), # Dummy vector size for demo
)

# Six KinkyShoeBros chunks with varied metadata.
sample_chunks = [
    {"id": 0, "text": "Retail managers receive Nike Air Max 95 quarterly.", "source": "policies.pdf",   "doc_type": "pdf",  "date": "2026-01-15", "year": 2026},
    {"id": 1, "text": "Level 1 Associates get Air Force 1s annually.", "source": "policies.pdf",   "doc_type": "pdf",  "date": "2026-01-15", "year": 2026},
    {"id": 2, "text": "Grade 10 deadstock pre-loved shoes yield 85% payout.", "source": "preloved.docx", "doc_type": "docx",  "date": "2026-01-15", "year": 2026},
    {"id": 3, "text": "Legacy Policy: All staff get generic canvas shoes.", "source": "old_policies.pdf", "doc_type": "pdf", "date": "2022-03-01", "year": 2022},
    {"id": 4, "text": "Shoe fitment guide: Nike Pegasus 40 for narrow feet.", "source": "catalog.html",   "doc_type": "html", "date": "2026-06-20", "year": 2026},
]

# Upsert with dummy vectors 
points = [
    PointStruct(id=c["id"], vector=np.random.rand(3).tolist(), payload={k: c[k] for k in ["text", "source", "doc_type", "date", "year"]})
    for c in sample_chunks
]
qdrant.upsert(collection_name=COLLECTION, points=points)
print(f"Upserted {len(points)} chunks with metadata.\n")

def show_hits(results, label):
    print(f"── {label} ──")
    for h in results:
        p = h.payload
        print(f"  {p['source']:18s} ({p['doc_type']}, {p['date']})  {p['text'][:50]}")
    print()

# Dummy query vector
q_vec = np.random.rand(3).tolist()

# Filter: only PDFs
results = qdrant.query_points(
    collection_name=COLLECTION, query=q_vec, limit=3,
    query_filter=Filter(must=[FieldCondition(key="doc_type", match=MatchValue(value="pdf"))]),
).points
show_hits(results, "Filter: doc_type = 'pdf'")

# Filter: only docs from 2026 or later (excluding legacy 2022 doc)
results = qdrant.query_points(
    collection_name=COLLECTION, query=q_vec, limit=3,
    query_filter=Filter(must=[FieldCondition(key="year", range=Range(gte=2026))]),
).points
show_hits(results, "Filter: year >= 2026 (exclude legacy 2022 doc)")

Upserted 5 chunks with metadata.

── Filter: doc_type = 'pdf' ──
  policies.pdf       (pdf, 2026-01-15)  Retail managers receive Nike Air Max 95 quarterly.
  policies.pdf       (pdf, 2026-01-15)  Level 1 Associates get Air Force 1s annually.
  old_policies.pdf   (pdf, 2022-03-01)  Legacy Policy: All staff get generic canvas shoes.

── Filter: year >= 2026 (exclude legacy 2022 doc) ──
  policies.pdf       (pdf, 2026-01-15)  Retail managers receive Nike Air Max 95 quarterly.
  preloved.docx      (docx, 2026-01-15)  Grade 10 deadstock pre-loved shoes yield 85% payou
  catalog.html       (html, 2026-06-20)  Shoe fitment guide: Nike Pegasus 40 for narrow fee



---

## Cell 4 — Why PII in vector stores is a problem

**Vector stores are search indexes.** Anything you put in them is discoverable by anyone with query access. If your ingested chunks contain PII (like HR contact info), that PII is one query away from being surfaced.

In [5]:
leaky_chunk = (
    "For questions regarding shoe allocations, contact HR Partner Sarah Chen. "
    "Her email is sarah.chen@kinkyshoebros.com and her employee ID is EMP-01847."
)
print(f"LEAKED CHUNK:\n{leaky_chunk}\n")
print("If this goes into Qdrant, a prompt like 'Who handles HR?' surfaces Sarah's private Employee ID.")
print("The rule: redact BEFORE you ingest, not after.")

LEAKED CHUNK:
For questions regarding shoe allocations, contact HR Partner Sarah Chen. Her email is sarah.chen@kinkyshoebros.com and her employee ID is EMP-01847.

If this goes into Qdrant, a prompt like 'Who handles HR?' surfaces Sarah's private Employee ID.
The rule: redact BEFORE you ingest, not after.


---

## Cells 5 to 7 — The Three Legs of PII Scrubbing

**No single tool catches everything.** Production pipelines use three legs:
1. **Regex** — fast; catches obvious patterns (emails, phones, IDs).
2. **Presidio (NER)** — catches names, organizations, locations; higher recall.
3. **Manual audit** — human eyes on a sample of ingested chunks.

Let's build a hybrid scrubber.

In [6]:
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine

analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()

PII_PATTERNS = {
    "EMAIL": re.compile(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'),
    "PHONE": re.compile(r'\+?\d{1,3}[-.\s]?\(?\d{1,4}\)?[-.\s]?\d{1,4}[-.\s]?\d{1,9}'),
    "EMP_ID": re.compile(r'EMP-\d{5}')
}

def scrub_pii_regex(text: str) -> tuple[str, list[str]]:
    flags = []
    scrubbed = text
    for label, pattern in PII_PATTERNS.items():
        for match in pattern.finditer(scrubbed):
            flags.append(f"{label}: {match.group(0)}")
        scrubbed = pattern.sub(f"<{label}>", scrubbed)
    return scrubbed, flags

def scrub_pii_hybrid(text: str) -> tuple[str, list[dict]]:
    flags = []
    # Leg 1: Regex
    scrubbed, regex_flags = scrub_pii_regex(text)
    for f in regex_flags:
        entity_type, matched = f.split(": ", 1)
        flags.append({"source": "regex", "entity_type": entity_type, "matched_text": matched, "score": 1.0})
    
    # Leg 2: Presidio NER
    results = analyzer.analyze(text=scrubbed, language="en")
    if results:
        anon = anonymizer.anonymize(text=scrubbed, analyzer_results=results)
        scrubbed = anon.text
        for r in results:
            flags.append({"source": "presidio", "entity_type": r.entity_type, "score": r.score})
            
    return scrubbed, flags

# Test it!
test_text = (
    "Contact Sarah Chen (HR Business Partner) at sarah.chen@kinkyshoebros.com or "
    "+1-555-0142. Her employee ID is EMP-01847. You can also reach Michael "
    "Torres at michael.torres@kinkyshoebros.com."
)

scrubbed_final, detected = scrub_pii_hybrid(test_text)

print(f"Original:\n  {test_text}\n")
print(f"Scrubbed:\n  {scrubbed_final}\n")
print(f"Total PII entities caught: {len(detected)}")
for f in detected:
    print(f"  - {f['source'].upper()}: {f['entity_type']}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 62.8 MB/s eta 0:00:00m eta 0:00:010:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Original:
  Contact Sarah Chen (HR Business Partner) at sarah.chen@kinkyshoebros.com or +1-555-0142. Her employee ID is EMP-01847. You can also reach Michael Torres at michael.torres@kinkyshoebros.com.

Scrubbed:
  Contact <PERSON> (HR Business Partner) at <EMAIL> or <PHONE>. Her employee ID is EMP-<PHONE>. You can also reach <PERSON> at <EMAIL>.

Total PII entities caught: 6
  - REGEX: EMAIL
  - REGEX: EMAIL
  - REGEX: PHONE
  - REGEX: PHONE
  - PRESIDIO: PERSON
  - PRESIDIO: PERSON


---

## Cell 8 — Wire it together: Mini-pipeline on one document

**The synthesis moment.** Take the KinkyShoeBros PDF, run it end-to-end through:
1. Parse
2. Chunk
3. Scrub PII (Hybrid Regex + Presidio)
4. Attach Metadata
5. Print the final list of ingestion-ready chunks

In [7]:
def ingest_one_pdf(path: Path) -> list[dict]:
    text = parse_pdf(path)
    chunk_texts = chunk_recursive(text, max_size=400)
    
    ingested_at = datetime.now(timezone.utc).isoformat()
    doc_stem = path.stem 
    
    ingested = []
    for idx, raw_text in enumerate(chunk_texts):
        scrubbed, flags = scrub_pii_hybrid(raw_text)
        
        chunk = {
            "chunk_id":         f"{doc_stem}#{idx}",
            "source":           path.name,
            "doc_type":         path.suffix.lstrip(".").lower(),
            "section_path":     "General",
            "page":             None,
            "date":             "2026-10-10",
            "language":         "en",
            "version":          "v1",
            "ingested_at":      ingested_at,
            "text":             scrubbed,
            "pii_flags_count":  len(flags),
        }
        ingested.append(chunk)
    
    return ingested

chunks = ingest_one_pdf(SAMPLE_PDF)

print(f"Ingested {SAMPLE_PDF.name} → {len(chunks)} chunks\n")
print(f"Total PII items scrubbed across doc: {sum(c['pii_flags_count'] for c in chunks)}\n")

pii_chunks = [c for c in chunks if c['pii_flags_count'] > 0]
if pii_chunks:
    print(f"══ Example of a scrubbed chunk (had {pii_chunks[0]['pii_flags_count']} PII items) ══")
    print(f"  {pii_chunks[0]['text']}")
    print("\n  Notice: emails, phones, IDs, and names are replaced with placeholders.")
    print("  The chunk is still semantically useful for retrieval, but the PII is gone!")

Ingested kinkyshoebros_policies.pdf → 3 chunks

Total PII items scrubbed across doc: 11

══ Example of a scrubbed chunk (had 1 PII items) ══
  kinkyShoeBros — Internal Employee Handbook
Section 1: Employee Footwear Allocation Program
As representatives of kinkyShoeBros, retail and corporate employees are required to wear approved
footwear during operating hours. The company provides a <DATE_TIME> stipend and direct allocations to
ensure staff are outfitted in the latest Nike, adidas, and New Balance releases.

  Notice: emails, phones, IDs, and names are replaced with placeholders.
  The chunk is still semantically useful for retrieval, but the PII is gone!
